# GRPO v2 Fine-tuning — `translategemma-4b-it` (ES → Valencian)

Loads the SFT checkpoint and runs GRPO with a **four-component composite reward**:
- **chrF** (w=0.5) — sentence-level chrF against reference; lexical/morphological fidelity
- **COMET** (w=0.3) — `wmt22-comet-da` semantic quality score (CPU to avoid OOM)
- **TTR** (w=0.2) — type-token ratio; penalises vocabulary collapse
- **copy penalty** — −1.0 if model returns the Spanish source untranslated

Published model: `guerreropaula/translategemma4b-grpo2-es-va`


## 1. Dependencies

In [ ]:
%%capture
!pip install -q transformers datasets accelerate peft trl bitsandbytes sacrebleu sentencepiece huggingface_hub unbabel-comet

In [ ]:
import torch, gc, os
import transformers, peft, trl
from typing import List

print(f"torch: {torch.__version__} | transformers: {transformers.__version__} | trl: {trl.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

## 2. Configuration

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""  # ← set your token
login(token=HF_TOKEN) if HF_TOKEN else login()

BASE_MODEL_ID = "google/translategemma-4b-it"
SFT_MODEL_ID  = "guerreropaula/translategemma4b-sft-es-va2"
GRPO_HUB_ID   = "guerreropaula/translategemma4b-grpo2-es-va"
OUTPUT_DIR    = "./translategemma4b_grpo_v2"

SOURCE_LANG_CODE = "es"
TARGET_LANG_CODE = "ca"
SOURCE_COL, TARGET_COL = "ES", "VA"

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_bf16_supported()
print(f"device: {DEVICE} | bf16: {USE_BF16}")

## 3. Load SFT Checkpoint

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)

# left-padding for batched GRPO generation
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)
base_model = prepare_model_for_kbit_training(base_model)
model      = PeftModel.from_pretrained(base_model, SFT_MODEL_ID, is_trainable=True)
model.print_trainable_parameters()

## 4. Prompt Template

In [ ]:
def _make_messages(source_text: str) -> list:
    return [{"role":"user","content":[{
        "type":SOURCE_LANG_CODE,"source_lang_code":SOURCE_LANG_CODE,
        "target_lang_code":TARGET_LANG_CODE,"text":source_text
    }]}]

def make_inference_prompt(source_text: str) -> str:
    return tokenizer.apply_chat_template(
        _make_messages(source_text), tokenize=False, add_generation_prompt=True
    )

print(make_inference_prompt("El ayuntamiento aprobó el presupuesto."))

## 5. GRPO Training Dataset

In [ ]:
from datasets import load_dataset

N_TRAIN = 10_000
raw     = load_dataset("gplsi/amic_parallel")

def preprocess(examples):
    return {
        "prompt"    : [make_inference_prompt(s) for s in examples["ES"]],
        "reference" : list(examples["VA"]),
        "source_es" : list(examples["ES"]),
    }

grpo_dataset = (
    raw["train"].shuffle(seed=42).select(range(N_TRAIN))
    .map(preprocess, batched=True, remove_columns=raw["train"].column_names)
)
print(grpo_dataset)
print("\nSample prompt:", grpo_dataset[0]["prompt"][:120], "...")

## 6. Load COMET (rank-0 only to avoid GPU OOM)

In [ ]:
import torch.distributed as dist
from comet import download_model, load_from_checkpoint
import logging
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

LOCAL_RANK     = int(os.environ.get("LOCAL_RANK", 0))
IS_DISTRIBUTED = dist.is_initialized()

comet_model = None
if LOCAL_RANK == 0:
    _path       = download_model("Unbabel/wmt22-comet-da")
    comet_model = load_from_checkpoint(_path).to("cuda")
    print("COMET loaded.")

if IS_DISTRIBUTED:
    dist.barrier()

## 7. Reward Functions

In [ ]:
import sacrebleu


def chrf_score(hypothesis: str, reference: str) -> float:
    """Sentence-level chrF normalised to [0, 1]."""
    if not hypothesis or not reference:
        return 0.0
    return sacrebleu.sentence_chrf(hypothesis, [reference]).score / 100.0


def ttr_score(hypothesis: str) -> float:
    """Type-token ratio. Short hypotheses (<5 tokens) are down-weighted."""
    if not hypothesis:
        return 0.0
    tokens = hypothesis.lower().split()
    if not tokens:
        return 0.0
    ttr = len(set(tokens)) / len(tokens)
    return ttr * min(1.0, len(tokens) / 5) if len(tokens) < 5 else ttr


def copy_penalty(source: str, hypothesis: str) -> float:
    """Returns -1.0 if hypothesis copies the source sentence verbatim, else 0.0."""
    if not hypothesis or not source:
        return 0.0
    return -1.0 if hypothesis.strip() == source.strip() else 0.0


def _comet_batch(sources: List[str], hyps: List[str], refs: List[str]) -> List[float]:
    """Run COMET on CPU on rank-0 and broadcast to other ranks."""
    if comet_model is None:
        return [0.5] * len(hyps)
    data   = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(sources, hyps, refs)]
    output = comet_model.predict(data, batch_size=8, gpus=0)
    scores = output.scores if hasattr(output, "scores") else output[0]

    if IS_DISTRIBUTED:
        t = torch.tensor(scores, dtype=torch.float32, device="cuda")
        dist.broadcast(t, src=0)
        scores = t.cpu().tolist()
    return scores


# Weights
W_CHRF, W_COMET, W_TTR = 0.5, 0.3, 0.2


def composite_reward(
    completions: List[str],
    reference:   List[str],
    source_es:   List[str],
    **kwargs,
) -> List[float]:
    """
    r = W_CHRF * chrF(hyp, ref)
      + W_COMET * COMET(src, hyp, ref)
      + W_TTR   * TTR(hyp)
      + copy_penalty(src, hyp)
    """
    comet_scores = _comet_batch(source_es, completions, reference)
    rewards = []
    for hyp, ref, src, cs in zip(completions, reference, source_es, comet_scores):
        hyp = hyp.strip() if isinstance(hyp, str) else ""
        r   = (W_CHRF  * chrf_score(hyp, ref)
             + W_COMET * cs
             + W_TTR   * ttr_score(hyp)
             + copy_penalty(src, hyp))
        rewards.append(r)
    return rewards


# Sanity check
test_hyps = ["L'ajuntament ha aprovat el nou pressupost.", "El ayuntamiento ha aprobado el nuevo presupuesto."]
test_refs = ["L'ajuntament ha aprovat el nou pressupost.", "L'ajuntament ha aprovat el nou pressupost."]
test_srcs = ["El ayuntamiento aprobó el presupuesto.", "El ayuntamiento ha aprobado el nuevo presupuesto."]
r = composite_reward(test_hyps, test_refs, test_srcs)
for h, score in zip(test_hyps, r):
    print(f"  {score:.4f}  {h[:60]}")

## 8. GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback
import matplotlib.pyplot as plt

class RewardPlotCallback(TrainerCallback):
    def __init__(self):
        self.steps, self.rewards = [], []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "reward" in logs:
            self.steps.append(state.global_step)
            self.rewards.append(logs["reward"])
            if len(self.steps) > 1:
                plt.figure(figsize=(10, 3)); plt.plot(self.steps, self.rewards, lw=1.5, color="#1D9E75")
                plt.title("GRPO v2 Reward"); plt.xlabel("step"); plt.ylabel("mean reward")
                plt.grid(alpha=0.25); plt.tight_layout(); plt.savefig("grpo_v2_reward.png", dpi=150); plt.close()

grpo_config = GRPOConfig(
    max_completion_length       = 128,
    num_generations             = 4,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 16,
    max_steps                   = 200,
    learning_rate               = 5e-6,
    warmup_steps                = 20,
    lr_scheduler_type           = "cosine",
    optim                       = "paged_adamw_8bit",
    weight_decay                = 0.01,
    beta                        = 0.04,
    epsilon                     = 0.2,
    bf16                        = USE_BF16,
    fp16                        = not USE_BF16,
    gradient_checkpointing      = True,
    output_dir                  = OUTPUT_DIR,
    logging_steps               = 10,
    save_steps                  = 50,
    report_to                   = "none",
    seed                        = 42,
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = composite_reward,
    args             = grpo_config,
    train_dataset    = grpo_dataset,
    callbacks        = [RewardPlotCallback()],
)

torch.cuda.empty_cache(); gc.collect()
stats = trainer.train()
print(f"Done — {stats.metrics['train_runtime']:.0f}s | VRAM peak: {torch.cuda.max_memory_reserved()/1e9:.2f} GB")

## 9. Save and Push to HuggingFace Hub

In [ ]:
# Save LoRA adapter
adapter_dir = OUTPUT_DIR + "/lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter saved: {adapter_dir}")

# Merge and push
merged_model = model.merge_and_unload()
merged_dir   = OUTPUT_DIR + "/merged"
merged_model.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
print(f"Merged model saved: {merged_dir}")

merged_model.push_to_hub(GRPO_HUB_ID, token=HF_TOKEN, safe_serialization=True, max_shard_size="2GB")
tokenizer.push_to_hub(GRPO_HUB_ID, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{GRPO_HUB_ID}")

## 10. Quick Evaluation Check

In [ ]:
from datasets import load_dataset as ld
import sacrebleu

N_QUICK = 100
test_ds = ld("gplsi/ES-VA_translation_test", split="test").select(range(N_QUICK))
gold_es = [ex["es"] for ex in test_ds]
gold_va = [ex["va"] for ex in test_ds]

merged_model.eval()
tokenizer.padding_side = "left"
hyps = []
for src in gold_es:
    prompt = make_inference_prompt(src)
    enc    = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = merged_model.generate(**enc, max_new_tokens=128, do_sample=False,
                                    pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    if "model" in text: text = text.split("model")[-1].strip()
    hyps.append(text)

chrf = sacrebleu.corpus_chrf(hyps, [gold_va]).score
bleu = sacrebleu.corpus_bleu(hyps, [gold_va]).score
print(f"Quick eval ({N_QUICK} sents) — chrF: {chrf:.2f} | BLEU: {bleu:.2f}")